# Final Notebook — Does More AI Mean More Output?
MGSC 661 · Group 3

- Sections 1–3 are the Proof of Concept (unchanged, except the post-PoC data fixes switched on below).
- Sections 4–9 are the post-PoC work behind the final presentation: statistical inference (robust OLS),
  hypothesis tests, diagnostics, hyperparameter tuning, a 9-model ML benchmark, and figures.
- Set `APPLY_POST_POC_FIXES = False` to reproduce the original PoC results (with the 23 h/day outlier).

# Modules & Parameters
- Required libraries and establishes shared settings for reproducibility.

In [ ]:
# Modules

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# Statistical Inferences
import statsmodels.api as sm

from pathlib import Path
from sklearn.model_selection import train_test_split
# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
# Cross Validation
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
# Model Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
# Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV

# Load Data #

path_dir = Path("AI_Usage_and_Impact_on_Students_and_Professionals.csv")    # CSV placeholder (CSV must be in working directory)

def load_dataset_data():
    dataset = pd.read_csv(path_dir)
    print(f"Dataset loaded: {path_dir}")                                    # path directory, dataset loaded correctly
    print(f"Rows: {dataset.shape[0]}, Columns: {dataset.shape[1]}")         # count rows/columns
    return dataset

dataset = load_dataset_data()

# Post-PoC additions
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

APPLY_POST_POC_FIXES = True   # False = reproduce the PoC exactly


In [ ]:
# Settings

## Data Cleaning
num_str_col = ["Age", "Monthly_Income", "AI_Usage_Hours_Per_Day", "Time_Saved_Hours_Per_Week", "Work_or_Study_Hours_Per_Day", "Monthly_AI_Cost"]
cat_col = ["Gender", "Country", "User_Type", "Education_Level", "Profession", "AI_Tool", "AI_Purpose", "Tasks_Performed", "Would_Recommend"]
## EDA
num_col = ["Age", "AI_Usage_Hours_Per_Day", "Monthly_AI_Cost", "Productivity_Score", "Accuracy_Rating", "Satisfaction_Score", "Time_Saved_Hours_Per_Week", "Work_or_Study_Hours_Per_Day"]

# Random State

rs = 69

# CV = Shuffled KFold

cv = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = rs
)

# Scoring

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

# Data Cleaning and EDA
- Prepares the dataset by correcting data types, standardizing variables, and creating indicators.

In [ ]:
# Removing Duplicates

dataset = dataset.drop_duplicates().copy()

# Converting Strings

for col in num_str_col:
    dataset[col] = (
        dataset[col]
        .astype("string")
        .str.replace(r"[^0-9.\-]", "", regex=True)
    )

    dataset[col] = pd.to_numeric(
        dataset[col],
        errors = "coerce"
    )

for col in cat_col:
    dataset[col] = dataset[col].astype("category")

dataset.describe().round(2)

# Creating Indicators for Monthly AI Cost (1-Paid vs 0-Free vs NA-Unknown)

dataset["Paid_AI"] = (
    dataset["Monthly_AI_Cost"] > 0
).astype("Int64")

dataset.loc[
    dataset["Monthly_AI_Cost"].isna(), "Paid_AI"
] = pd.NA

In [ ]:
# Standardizing Categorical Variables

dataset["Gender"] = dataset["Gender"].replace({
    "male": "Male",
    "M": "Male",
    "MALE": "Male",
    "female": "Female",
    "F": "Female",
    "FEMALE": "Female",
    "other": "Other"
})

dataset["Country"] = dataset["Country"].replace({
    "brazil": "Brazil",
    "india": "India",
    "canada": "Canada",
    "australia": "Australia",
    "france": "France",
    "philippines": "Philippines",
    "U.K.": "United Kingdom",
    "United States of America": "United States"
})

dataset["Education_Level"] = dataset["Education_Level"].replace({
    "undergrad": "Undergraduate",
    "BA/BSc": "Bachelor's",
    "Bachelors": "Bachelor's",
    "bachelor's degree": "Bachelor's",
    "Masters": "Master's",
    "MA/MSc": "Master's",
    "master's degree": "Master's",
    "Ph.D.": "PhD",
    "HS": "High School",
    "high school": "High School"
})

dataset["AI_Tool"] = dataset["AI_Tool"].replace({
    "chatgpt": "ChatGPT",
    "Chat GPT": "ChatGPT",
    "Chatgpt": "ChatGPT",
    "claude": "Claude",
    "Copilot": "Microsoft Copilot",
    "copilot": "Microsoft Copilot",
    "Gemini": "Google Gemini",
    "gemini": "Google Gemini"
})

# Validation (optional check)

#for col in cat_col:
    #print(f"\n{col}")
    #print(dataset[col].value_counts(dropna=False))

## Post-PoC data fixes
- **Implausible values → missing** (not deleting the row): AI usage or study > 16 h/day, age > 80, monthly income > 50,000.
  In the PoC, the single 23 h/day respondent drove linear CV R² below zero.
- `Is_Student` indicator for regression; `Paid_AI` converted to float so statsmodels can use it (missing stays missing).

In [ ]:
implausible_rules = {
    "AI_Usage_Hours_Per_Day": lambda d: d["AI_Usage_Hours_Per_Day"] > 16,
    "Work_or_Study_Hours_Per_Day": lambda d: d["Work_or_Study_Hours_Per_Day"] > 16,
    "Age": lambda d: d["Age"] > 80,
    "Monthly_Income": lambda d: d["Monthly_Income"] > 50000,
}

if APPLY_POST_POC_FIXES:
    print("Values set to missing")
    print("=" * 45)
    for col, rule in implausible_rules.items():
        mask = rule(dataset)
        print(f"{col}: {dataset.loc[mask, ['User_ID', col]].values.tolist()}")
        dataset.loc[mask, col] = np.nan

dataset["Is_Student"] = (dataset["User_Type"] == "Student").astype(float)
dataset["Paid_AI"] = dataset["Paid_AI"].astype("float")
print("\nRows:", len(dataset))

## EDA - Numerical Distributions
- Visualizes numerical variables using histograms and boxplots to identify patterns and outliers.

In [ ]:
# Histograms

dataset[num_col].hist(
    figsize= (14, 10),
    bins= 20
)
plt.tight_layout()
plt.show()

# Boxplots

fig, axes = plt.subplots(3, 3, figsize=(14, 10))

for col, ax in zip(num_col, axes.flatten()):
    ax.boxplot(dataset[col].dropna())
    ax.set_title(col)
    ax.set_ylabel("Value")

plt.tight_layout()
plt.show()

## EDA - Correlation Matrix
- Examines linear relationships between numerical variables and the target outcomes.

In [ ]:
corr_matrix = dataset[num_col].corr()

# Correlations with Productivity

productivity_corr = (
    corr_matrix["Productivity_Score"]
    .drop("Productivity_Score")
    .sort_values(ascending=False)
)

print("Correlations with Productivity:")
print("=" * 45)
print(productivity_corr.round(2))

# Correlations with Satisfaction

satisfaction_corr = (
    corr_matrix["Satisfaction_Score"]
    .drop("Satisfaction_Score")
    .sort_values(ascending=False)
)

print("\nCorrelations with Satisfaction:")
print("=" * 45)
print(satisfaction_corr.round(2))

## EDA - Categorical Variables
- Compares average productivity and satisfaction scores across categorical groups.

In [ ]:
# EDA - Categorical Variables vs Outcomes

model_cat_col = [
    "User_Type",
    "AI_Purpose",
    "AI_Tool",
    "Education_Level",
    "Gender"
]

for col in model_cat_col:
    summary = (
        dataset.groupby(col, observed=True)
        .agg(
            N=("Productivity_Score", "size"),
            Avg_Productivity=("Productivity_Score", "mean"),
            Avg_Satisfaction=("Satisfaction_Score", "mean")
        )
        .round(2)
    )

    print(f"\n=== {col} ===")
    display(summary)


## EDA - Paid vs Free Subscriptions
- Compares productivity and satisfaction outcomes between free and paid AI users.

In [ ]:
# Distribution

print("Paid vs Free Distribution")
print(dataset["Paid_AI"].value_counts(dropna= False))


# Summary

paid_summary = (
    dataset.groupby("Paid_AI", dropna= False)
    .agg(
        N= ("Productivity_Score", "size"),
        Avg_Productivity= ("Productivity_Score", "mean"),
        Avg_Satisfaction= ("Satisfaction_Score", "mean")
    )
    .round(2)
)

display(paid_summary)


# Model 01 - Productivity
- Develops and compares regression algorithms to predict productivity scores.

In [ ]:
# Preparing the Model

model01_col = ["Productivity_Score", "AI_Usage_Hours_Per_Day", "Paid_AI", "User_Type", "AI_Purpose", "Age"]
model01_ds = dataset[model01_col].copy()

print("Observations:", len(model01_ds))
print("\nMissing values:")
print("=" * 30)
print(model01_ds.isna().sum())

# Defining Xs & Y

model01_ds = model01_ds.dropna(subset= ["Productivity_Score"]).copy()
model01_X = model01_ds.drop(columns= ["Productivity_Score"])
model01_y = model01_ds["Productivity_Score"]

print("\nObservations:", len(model01_ds))
print("=" * 30)
print("X shape:", model01_X.shape)
print("y shape:", model01_y.shape)

# Train-Test Split (80:20)

X_train01, X_test01, y_train01, y_test01 = train_test_split(
    model01_X,
    model01_y,
    test_size = 0.20,
    random_state = rs
)

print("\nTrain-Test Split [80:20]")
print("=" * 30)
print("Model 1 - Training-set:", X_train01.shape[0])
print("Model 1 - Testing-set:", X_test01.shape[0])

In [ ]:
# Preprocessing

num_features = ["AI_Usage_Hours_Per_Day", "Age", "Paid_AI"]
cat_features = ["User_Type", "AI_Purpose"]

# Numerical - Mdedian Imputing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy= "median"))
])

# Categorical - Most-frequent Imputing + OneHot Encoding
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy= "most_frequent")),
    ("encoder", OneHotEncoder(
        drop= "first",
        handle_unknown= "ignore"
    ))
])

# Custom - Column Transformer
preprocessor01 = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

In [ ]:
# Algorithm Definitions

# Scaling for Ridge and Lasso
scaled_preprocessor01 = clone(preprocessor01)

scaled_preprocessor01.set_params(
    num__steps=[
        ("imputer", SimpleImputer(strategy= "median")),
        ("scaler", StandardScaler())
    ]
)

# Squared AI-usage feature
quad_preprocessor = ColumnTransformer([
    ("base", clone(preprocessor01), [
        "AI_Usage_Hours_Per_Day",
        "Age",
        "Paid_AI",
        "User_Type",
        "AI_Purpose"
    ]),

    ("usage_squared", Pipeline([
        ("imputer", SimpleImputer(strategy= "median")),
        ("square", FunctionTransformer(
            lambda x: x ** 2,
            feature_names_out= "one-to-one"
        ))
    ]), ["AI_Usage_Hours_Per_Day"])
])

# Algorithm Definitions
models01 = {
    # Multiple Linear Regression
    "Linear": Pipeline([
        ("preprocessor", clone(preprocessor01)),
        ("regressor", LinearRegression())
    ]),
    # Quadratic Regression
    "Quadratic": Pipeline([
        ("preprocessor", clone(quad_preprocessor)),
        ("regressor", LinearRegression())
    ]),
    # Ridge Regression (L2 Regularization)
    "Ridge": Pipeline([
        ("preprocessor", clone(scaled_preprocessor01)),
        ("regressor", Ridge(alpha= 1.0))
    ]),
    # Lasso Regression (L1)
    "Lasso": Pipeline([
        ("preprocessor", clone(scaled_preprocessor01)),
        ("regressor", Lasso(alpha= 0.1, max_iter= 10000))
    ]),
    # Random Forest Regression
    "Random Forest": Pipeline([
        ("preprocessor", clone(preprocessor01)),
        ("regressor", RandomForestRegressor(
            n_estimators= 200,
            min_samples_leaf= 3,
            random_state= rs,
            n_jobs= -1
        ))
    ])
}

# Algorithm Comparison - Debugging

results01 = {}
print("\nAlgorithm Testing")
print("=" * 20)

for name, model in models01.items():
    print(f"Testing: {name}")

    results01[name] = cross_validate(
        model,
        X_train01,
        y_train01,
        cv= cv,
        scoring= scoring,
        error_score= "raise"
    )

    print("Passed")

# Cross Validation

comparison01 = pd.DataFrame({
    name: {
        "MAE": -res["test_MAE"].mean(),
        "RMSE": -res["test_RMSE"].mean(),
        "R2": res["test_R2"].mean()
    }
    for name, res in results01.items()
}).T

print("\nModel 1 - Productivity: KFold - 5 Fold CV")
display(comparison01.sort_values("RMSE").round(3))

Interpretation of Outputs (after the post-PoC fixes):
- PoC (23 h/day row included): Linear CV R² = -0.079, Quadratic = 0.528. The single impossible observation broke every linear model.
- After setting implausible values to missing: Linear R² = 0.528, Quadratic = 0.580 (still best), Random Forest = 0.559.
- So the outlier, not nonlinearity, caused most of the PoC gap; the remaining curvature is tested formally in Section 4.

# Model 02 - Satisfaction
- Develops and compares regression algorithms to predict satisfaction scores.

In [ ]:
# Preparing the Model

model02_col = ["Satisfaction_Score", "Accuracy_Rating", "Productivity_Score", "AI_Usage_Hours_Per_Day", "Paid_AI", "User_Type", "Age"]
model02_ds = dataset[model02_col].copy()

print("Observations:", len(model02_ds))
print("\nMissing Values:")
print("=" * 30)
print(model02_ds.isna().sum())

# Defining Xs & Y

model02_ds = model02_ds.dropna(subset = ["Satisfaction_Score"]).copy()
model02_X = model02_ds.drop(columns=["Satisfaction_Score"])
model02_y = model02_ds["Satisfaction_Score"]

print("\nObservations:", len(model02_ds))
print("=" * 30)
print("X shape:", model02_X.shape)
print("Y shape:", model02_y.shape)

# Train-Test Split (80:20)

X_train02, X_test02, y_train02, y_test02 = train_test_split(
    model02_X,
    model02_y,
    test_size= 0.20,
    random_state= rs
)

print("\nTrain-Test Split [80:20]")
print("=" * 30)
print("Model 2 - Training-set:", X_train02.shape[0])
print("Model 2 - Testing-set:", X_test02.shape[0])

In [ ]:
# Preprocessing

num_features02 = ["Accuracy_Rating", "Productivity_Score", "AI_Usage_Hours_Per_Day", "Age"]
cat_features02 = ["User_Type"]
bin_features02 = ["Paid_AI"]

# Numerical - Mdedian Imputing
num_pipeline02 = Pipeline([
    ("imputer", SimpleImputer(strategy= "median"))
])

# Categorical - Most-frequent Imputing + OneHot Encoding
cat_pipeline02 = Pipeline([
    ("imputer", SimpleImputer(strategy= "most_frequent")),
    ("encoder", OneHotEncoder(
        drop= "first",
        handle_unknown= "ignore"
    ))
])

# Binary - Most-frequent Imputing
bin_pipeline02 = Pipeline([
    ("imputer", SimpleImputer(strategy= "most_frequent"))
])

# Custom - Column Transformer
preprocessor02 = ColumnTransformer([
    ("num", num_pipeline02, num_features02),
    ("cat", cat_pipeline02, cat_features02),
    ("bin", bin_pipeline02, bin_features02)
])

In [ ]:
# Algorithm Definitions

# Scaling for Ridge and Lasso
scaled_preprocessor02 = clone(preprocessor02)

scaled_preprocessor02.set_params(
    num__steps=[
        ("imputer", SimpleImputer(strategy= "median")),
        ("scaler", StandardScaler())
    ]
)

# Defining Models (Based on Model 1)
models02 = {
    # Multiple Linear Regression
    "Linear": Pipeline([
        ("preprocessor", clone(preprocessor02)),
        ("regressor", LinearRegression())
    ]),
    # Ridge Regression (L2 Regularization)
    "Ridge": Pipeline([
        ("preprocessor", clone(scaled_preprocessor02)),
        ("regressor", Ridge(alpha= 1.0))
    ]),
    # Lasso Regression (L1)
    "Lasso": Pipeline([
        ("preprocessor", clone(scaled_preprocessor02)),
        ("regressor", Lasso(alpha= 0.1, max_iter= 10000))
    ]),
    # Random Forest Regression
    "Random Forest": Pipeline([
        ("preprocessor", clone(preprocessor02)),
        ("regressor", RandomForestRegressor(
            n_estimators= 200,
            min_samples_leaf= 3,
            random_state= rs,
            n_jobs= -1
        ))
    ])
}

# Algorithm Comparison - Debugging
results02 = {}
print("\nAlgorithm Testing")
print("=" * 20)

for name, model in models02.items():
    print(f"Testing: {name}")

    results02[name] = cross_validate(
        model,
        X_train02,
        y_train02,
        cv= cv,
        scoring= scoring,
        error_score= "raise"
    )

    print("Passed")

# Cross Validation

comparison02 = pd.DataFrame({
    name: {
        "MAE": -res["test_MAE"].mean(),
        "RMSE": -res["test_RMSE"].mean(),
        "R2": res["test_R2"].mean()
    }
    for name, res in results02.items()
}).T

print("\nModel 2 - Satisfaction: KFold - 5 Fold CV")
display(comparison02.sort_values("RMSE").round(3))

Interpretation of Outputs:
- Linear, Ridge and Lasso are practically identical (CV R² ≈ 0.60); Random Forest is worse.
- Multiple linear regression is kept for interpretability; tuned comparison and test-set results are in Sections 7–8.

# 4. Statistical Inference — Robust OLS (HC3)
- Complete cases; heteroskedasticity-robust HC3 standard errors.
- AI usage is **centered at its mean** before squaring, so the linear term is the effect at average usage
  and the two terms are less collinear.

In [ ]:
def ols_design(df, y, num, cat=(), extra=None):
    """Complete-case design matrix with treatment dummies (alphabetically first level = reference)."""
    cols = [y] + list(num) + list(cat)
    d = df[cols].dropna().copy()
    X = d[list(num)].astype(float)
    for c in cat:
        X = X.join(pd.get_dummies(d[c].astype(str), prefix=c, drop_first=True, dtype=float))
    if extra:
        for name, fn in extra.items():
            X[name] = fn(d).astype(float)
    return d[y].astype(float), sm.add_constant(X), d


def robust_ols(y, X):
    return sm.OLS(y, X).fit(cov_type="HC3")


def joint_wald(res, terms):
    """Joint robust Wald F-test that all listed coefficients are zero."""
    R = np.zeros((len(terms), len(res.params)))
    for i, t in enumerate(terms):
        R[i, res.params.index.get_loc(t)] = 1
    w = res.wald_test(R, use_f=True, scalar=True)
    return float(np.squeeze(w.fvalue)), float(np.squeeze(w.pvalue))


def coef_table(res, terms=None):
    t = pd.DataFrame({"coef": res.params, "robust_se": res.bse, "t": res.tvalues, "p": res.pvalues})
    ci = res.conf_int(); t["ci_low"], t["ci_high"] = ci[0], ci[1]
    return (t if terms is None else t.loc[terms]).round(3)


usage_mean = dataset["AI_Usage_Hours_Per_Day"].mean()
dataset["Usage_c"] = dataset["AI_Usage_Hours_Per_Day"] - usage_mean
dataset["Usage_c2"] = dataset["Usage_c"] ** 2
print(f"Mean AI usage (centering constant): {usage_mean:.3f} h/day")

## 4.1 Model 1 — Productivity (quadratic, final)

In [ ]:
m1_num = ["Usage_c", "Usage_c2", "Paid_AI", "Is_Student", "Age"]
y1, X1, d1 = ols_design(dataset, "Productivity_Score", m1_num, ["AI_Purpose"])
ols1 = robust_ols(y1, X1)
print(f"n = {int(ols1.nobs)}, R2 = {ols1.rsquared:.3f}, adj. R2 = {ols1.rsquared_adj:.3f}")
display(coef_table(ols1))

F, p = joint_wald(ols1, ["Usage_c", "Usage_c2"])
print(f"H1 joint Wald (usage + usage^2): F = {F:.1f}, p = {p:.2e}")
purpose_terms = [c for c in X1.columns if c.startswith("AI_Purpose_")]
F, p = joint_wald(ols1, purpose_terms)
print(f"AI purpose dummies jointly:      F = {F:.2f}, p = {p:.3f}")

### Diminishing returns: marginal effect of one more hour

In [ ]:
b1, b2 = ols1.params["Usage_c"], ols1.params["Usage_c2"]
plateau = usage_mean - b1 / (2 * b2)
marginal = pd.DataFrame({"hours_per_day": [1, 2, 4, 6, 8]})
marginal["gain_per_extra_hour"] = b1 + 2 * b2 * (marginal["hours_per_day"] - usage_mean)
display(marginal.round(2))
print(f"Plateau (gains stop) at about {plateau:.1f} h/day; "
      f"max observed usage = {dataset['AI_Usage_Hours_Per_Day'].max():.1f} h/day")

### Model 1 robustness: linear form, interactions (H3/H4) and extra controls

In [ ]:
# Linear-only specification
y1l, X1l, _ = ols_design(dataset, "Productivity_Score", ["AI_Usage_Hours_Per_Day", "Paid_AI", "Is_Student", "Age"], ["AI_Purpose"])
ols1_lin = robust_ols(y1l, X1l)
print(f"Linear spec: R2 = {ols1_lin.rsquared:.3f} vs quadratic {ols1.rsquared:.3f}")

# H4: does the return to usage differ for students?
y, X, _ = ols_design(dataset, "Productivity_Score", m1_num, ["AI_Purpose"],
                     extra={"Student_x_Usage": lambda d: d["Is_Student"] * d["Usage_c"]})
print("\nStudent x usage interaction"); display(coef_table(robust_ols(y, X), ["Usage_c", "Is_Student", "Student_x_Usage"]))

# H3: does the return to usage differ for paid users?
y, X, _ = ols_design(dataset, "Productivity_Score", m1_num, ["AI_Purpose"],
                     extra={"Paid_x_Usage": lambda d: d["Paid_AI"] * d["Usage_c"]})
print("Paid x usage interaction"); display(coef_table(robust_ols(y, X), ["Usage_c", "Paid_AI", "Paid_x_Usage"]))

# Additional controls planned in the PoC
y1c, X1c, _ = ols_design(dataset, "Productivity_Score", m1_num + ["Work_or_Study_Hours_Per_Day"],
                         ["AI_Purpose", "AI_Tool", "Education_Level"])
ols1_ctrl = robust_ols(y1c, X1c)
print(f"With controls: n = {int(ols1_ctrl.nobs)}, R2 = {ols1_ctrl.rsquared:.3f}")
display(coef_table(ols1_ctrl, m1_num + ["Work_or_Study_Hours_Per_Day"]))
for prefix in ["AI_Tool_", "Education_Level_"]:
    F, p = joint_wald(ols1_ctrl, [c for c in X1c.columns if c.startswith(prefix)])
    print(f"{prefix[:-1]} jointly: F = {F:.2f}, p = {p:.3f}")

## 4.2 Model 2 — Satisfaction (linear, final)
`Time_Saved_Hours_Per_Week` is excluded from both models: it correlates ~0.90 with usage and is an outcome of usage (a mediator).

In [ ]:
print("corr(time saved, usage) =", round(dataset["Time_Saved_Hours_Per_Week"].corr(dataset["AI_Usage_Hours_Per_Day"]), 2))

m2_num = ["Accuracy_Rating", "Productivity_Score", "AI_Usage_Hours_Per_Day", "Paid_AI", "Is_Student", "Age"]
y2, X2, d2 = ols_design(dataset, "Satisfaction_Score", m2_num)
ols2 = robust_ols(y2, X2)
print(f"n = {int(ols2.nobs)}, R2 = {ols2.rsquared:.3f}, adj. R2 = {ols2.rsquared_adj:.3f}")
display(coef_table(ols2))

# Standardized betas (comparable effect sizes)
z = lambda s: (s - s.mean()) / s.std()
std_res = sm.OLS(z(y2), sm.add_constant(d2[m2_num].astype(float).apply(z))).fit()
display(std_res.params.drop("const").round(3).rename("standardized_beta").to_frame())

# Accuracy x productivity interaction (centered)
y, X, _ = ols_design(dataset, "Satisfaction_Score", m2_num, extra={
    "Acc_x_Prod": lambda d: (d["Accuracy_Rating"] - d["Accuracy_Rating"].mean()) * (d["Productivity_Score"] - d["Productivity_Score"].mean())})
display(coef_table(robust_ols(y, X), ["Accuracy_Rating", "Productivity_Score", "Acc_x_Prod"]))
print("corr(accuracy, productivity) =", round(dataset["Accuracy_Rating"].corr(dataset["Productivity_Score"]), 2))

# 5. Hypothesis Scorecard (α = 0.05)

In [ ]:
scorecard = pd.DataFrame([
    ["H1", "AI usage -> productivity", "Usage_c (+ Usage_c2)", ols1.params["Usage_c"], joint_wald(ols1, ["Usage_c", "Usage_c2"])[1]],
    ["H2", "Accuracy -> satisfaction", "Accuracy_Rating", ols2.params["Accuracy_Rating"], ols2.pvalues["Accuracy_Rating"]],
    ["H3", "Paid vs free -> productivity", "Paid_AI", ols1.params["Paid_AI"], ols1.pvalues["Paid_AI"]],
    ["H4", "Student vs professional -> productivity", "Is_Student", ols1.params["Is_Student"], ols1.pvalues["Is_Student"]],
], columns=["ID", "Hypothesis", "Term", "Estimate", "p_value"])
scorecard["Verdict"] = np.where(scorecard["p_value"] < 0.05, "Supported", "Not supported")
display(scorecard.round(4))

# 6. Diagnostics and Bivariate Tests

In [ ]:
def diagnostics(res, X):
    bp_p = het_breuschpagan(res.resid, X)[1]
    sw_p = stats.shapiro(res.resid).pvalue
    vif = max(variance_inflation_factor(X.values, i) for i in range(1, X.shape[1]))
    cooks = res.get_influence().cooks_distance[0].max()
    return {"Breusch-Pagan p": bp_p, "Shapiro-Wilk p": sw_p, "Max VIF": vif, "Max Cook's D": cooks}

display(pd.DataFrame({"Model 1": diagnostics(ols1, X1), "Model 2": diagnostics(ols2, X2)}).round(3))

# Bivariate tests
g = dataset.dropna(subset=["Productivity_Score", "Paid_AI"])
print("Paid vs free (Welch t):", stats.ttest_ind(g[g.Paid_AI == 1].Productivity_Score, g[g.Paid_AI == 0].Productivity_Score, equal_var=False))
g = dataset.dropna(subset=["Productivity_Score"])
print("Student vs professional (Welch t):", stats.ttest_ind(g[g.Is_Student == 1].Productivity_Score, g[g.Is_Student == 0].Productivity_Score, equal_var=False))
g = dataset.dropna(subset=["Productivity_Score", "AI_Purpose"])
groups = [x.Productivity_Score for _, x in g.groupby("AI_Purpose", observed=True)]
print("AI purpose (ANOVA):", stats.f_oneway(*groups))
print("AI purpose (Kruskal-Wallis):", stats.kruskal(*groups))
print("Mean usage, free vs paid:", dataset.groupby("Paid_AI").AI_Usage_Hours_Per_Day.mean().round(2).to_dict())

# 7. Hyperparameter Tuning and ML Benchmark
- GridSearchCV with the same shuffled 5-fold CV, on the training set only.
- Four extra ML models: gradient boosting, SVR (RBF), KNN, neural net (MLP). Scaled inputs for distance/kernel/NN models.
- This cell takes a few minutes.

In [ ]:
def tune_and_score(models, grids, X_tr, y_tr):
    tuned, rows = {}, {}
    for name, model in models.items():
        if name in grids:
            gs = GridSearchCV(model, grids[name], cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_tr, y_tr)
            model, best = gs.best_estimator_, {k.split("__")[-1]: v for k, v in gs.best_params_.items()}
        else:
            best = "-"
        res = cross_validate(model, X_tr, y_tr, cv=cv, scoring=scoring)
        tuned[name] = model.fit(X_tr, y_tr)
        rows[name] = {"CV_MAE": -res["test_MAE"].mean(), "CV_RMSE": -res["test_RMSE"].mean(),
                      "CV_R2": res["test_R2"].mean(), "best_params": str(best)}
        print(f"{name}: done")
    return tuned, pd.DataFrame(rows).T


def extra_ml(scaled_pre):
    return {
        "Gradient Boosting": Pipeline([("preprocessor", clone(scaled_pre)), ("regressor", GradientBoostingRegressor(random_state=rs))]),
        "SVR (RBF)": Pipeline([("preprocessor", clone(scaled_pre)), ("regressor", SVR())]),
        "KNN": Pipeline([("preprocessor", clone(scaled_pre)), ("regressor", KNeighborsRegressor())]),
        "Neural net (MLP)": Pipeline([("preprocessor", clone(scaled_pre)), ("regressor", MLPRegressor(max_iter=3000, early_stopping=True, random_state=rs))]),
    }


grids = {
    "Ridge": {"regressor__alpha": np.logspace(-3, 3, 25)},
    "Lasso": {"regressor__alpha": np.logspace(-4, 0, 25)},
    "Random Forest": {"regressor__n_estimators": [200, 500], "regressor__max_depth": [None, 4, 8],
                      "regressor__min_samples_leaf": [1, 3, 5, 10], "regressor__max_features": [1.0, 0.5]},
    "Gradient Boosting": {"regressor__n_estimators": [100, 300], "regressor__learning_rate": [0.03, 0.1],
                          "regressor__max_depth": [2, 3], "regressor__min_samples_leaf": [5, 10], "regressor__subsample": [0.8, 1.0]},
    "SVR (RBF)": {"regressor__C": [0.3, 1, 3, 10, 30], "regressor__epsilon": [0.1, 0.3, 0.5], "regressor__gamma": ["scale", 0.03, 0.1]},
    "KNN": {"regressor__n_neighbors": [5, 10, 15, 20, 30], "regressor__weights": ["uniform", "distance"]},
    "Neural net (MLP)": {"regressor__hidden_layer_sizes": [(8,), (16,), (16, 8)], "regressor__alpha": [0.001, 0.01, 0.1, 1.0]},
}

all_models01 = {**models01, **extra_ml(scaled_preprocessor01)}
tuned01, cv01 = tune_and_score(all_models01, grids, X_train01, y_train01)

all_models02 = {**models02, **extra_ml(scaled_preprocessor02)}
tuned02, cv02 = tune_and_score(all_models02, grids, X_train02, y_train02)

# 8. Final Evaluation on the Held-out Test Set

In [ ]:
def test_scores(tuned, cv_table, X_te, y_te, y_tr):
    out = cv_table.copy()
    for name, model in tuned.items():
        p = model.predict(X_te)
        out.loc[name, "Test_MAE"] = mean_absolute_error(y_te, p)
        out.loc[name, "Test_RMSE"] = root_mean_squared_error(y_te, p)
        out.loc[name, "Test_R2"] = r2_score(y_te, p)
    base = root_mean_squared_error(y_te, np.full(len(y_te), y_tr.mean()))
    print(f"Baseline test RMSE (predict training mean): {base:.3f}")
    num_cols = ["CV_MAE", "CV_RMSE", "CV_R2", "Test_MAE", "Test_RMSE", "Test_R2"]
    out[num_cols] = out[num_cols].astype(float)
    return out.sort_values("Test_RMSE")[num_cols + ["best_params"]]

final01 = test_scores(tuned01, cv01, X_test01, y_test01, y_train01)
print("Model 1 - Productivity"); display(final01.round(3))
final02 = test_scores(tuned02, cv02, X_test02, y_test02, y_train02)
print("Model 2 - Satisfaction"); display(final02.round(3))

Interpretation:
- Productivity: the quadratic regression has the lowest test RMSE; tuned random forest and gradient boosting are close
  in CV but not better on the test set, so the interpretable quadratic is kept.
- Satisfaction: Linear, Ridge and Lasso are tied; every flexible ML model is worse. Plain linear regression is kept.

In [ ]:
# Permutation importance (model-agnostic check against the OLS conclusions)
for label, model, X_te, y_te in [("Model 1 - Gradient Boosting", tuned01["Gradient Boosting"], X_test01, y_test01),
                                 ("Model 1 - Random Forest", tuned01["Random Forest"], X_test01, y_test01),
                                 ("Model 2 - Gradient Boosting", tuned02["Gradient Boosting"], X_test02, y_test02)]:
    pi = permutation_importance(model, X_te, y_te, n_repeats=30, random_state=rs)
    print(label); display(pd.Series(pi.importances_mean, index=X_te.columns).sort_values(ascending=False).round(3).to_frame("importance"))

# 9. Figures for the Presentation

In [ ]:
# Figure: diminishing returns (fitted curve + 95% CI, other predictors at their means)
hours = np.linspace(dataset["AI_Usage_Hours_Per_Day"].min(), dataset["AI_Usage_Hours_Per_Day"].max(), 200)
grid = pd.DataFrame(np.tile(X1.mean().values, (len(hours), 1)), columns=X1.columns)
grid["Usage_c"] = hours - usage_mean
grid["Usage_c2"] = grid["Usage_c"] ** 2
pred = ols1.get_prediction(grid).summary_frame(alpha=0.05)

fig, ax = plt.subplots(figsize=(10, 5.6))
ax.scatter(d1["Usage_c"] + usage_mean, y1, s=30, alpha=0.35, color="#2a78d6", edgecolor="white")
ax.fill_between(hours, pred["mean_ci_lower"], pred["mean_ci_upper"], color="#eb6834", alpha=0.18, lw=0)
ax.plot(hours, pred["mean"], color="#eb6834", lw=2.5)
ax.set_xlabel("AI usage (hours per day)"); ax.set_ylabel("Productivity score (1-10)")
ax.set_title("More AI -> more output, with diminishing returns", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("fig_diminishing_returns.png", dpi=200); plt.show()

# Figure: hypothesis scorecard (coefficients with 95% robust CIs)
rows = [("H1 Usage -> productivity", ols1, "Usage_c"), ("H2 Accuracy -> satisfaction", ols2, "Accuracy_Rating"),
        ("H3 Paid -> productivity", ols1, "Paid_AI"), ("H4 Student -> productivity", ols1, "Is_Student")]
fig, ax = plt.subplots(figsize=(10, 4.5))
for i, (lab, res, term) in enumerate(rows[::-1]):
    lo, hi = res.conf_int().loc[term]
    col = "#2a78d6" if res.pvalues[term] < 0.05 else "#9a9993"
    ax.plot([lo, hi], [i, i], color=col, lw=2.5); ax.scatter(res.params[term], i, s=90, color=col, zorder=3)
ax.axvline(0, color="#52514e", lw=1, ls="--")
ax.set_yticks(range(len(rows)), [r[0] for r in rows[::-1]])
ax.set_xlabel("Effect on score (points), 95% robust CI")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("fig_hypotheses.png", dpi=200); plt.show()

# 10. Conclusions
- **H1 supported:** usage strongly predicts productivity with diminishing returns (gain per extra hour falls from ~1.1 at 1 h/day to ~0.1 at 8 h/day; plateau ≈ 8.6 h/day).
- **H2 supported:** perceived accuracy predicts satisfaction (+0.86 per point) controlling for productivity; the two are nearly uncorrelated.
- **H3, H4 not supported:** paid status and user type show no productivity difference, and no different return per hour.
- Interpretable models match or beat all tuned ML models on the test set; permutation importance agrees with OLS.
- Limitations: synthetic, self-reported, cross-sectional data — associations, not causal effects.